# PPE compliance analysis — helmet + vest
Turns each model's test detections into an image-level decision: **compliant vs violation**, then reports accuracy, precision, recall, F1, ROC/PR curves and confusion matrices for every model and split.

**Rule:** ground truth = *violation* if the labels contain `no-helmet` or `no-vest`; *compliant* if there is a helmet/vest and no violation label (other images are ignored). Prediction = *violation alert* if the model detects `no-helmet` or `no-vest` with confidence ≥ `CONF_THRESHOLD`. Positive class = violation.

Needs the **updated training notebook** (it saves `test_predictions.csv` and `test_ground_truth.csv`). No GPU needed.

In [ ]:
# ============================================================
# PPE COMPLIANCE ANALYSIS: helmet + vest   (CPU only, run after the training notebook)
# Question answered: "Is this image COMPLIANT or does it contain a VIOLATION?"
#   Ground truth : VIOLATION if the labels contain no-helmet or no-vest,
#                  COMPLIANT if there is a helmet/vest and no violation label (other images are ignored).
#   Prediction   : VIOLATION alert if the model detects no-helmet or no-vest with confidence >= CONF_THRESHOLD.
#   Positive class = violation.
# ============================================================
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (confusion_matrix, roc_curve, roc_auc_score,
                             precision_recall_curve, average_precision_score)

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

RUNS_ROOT = Path("/content/drive/MyDrive/PPE_Safety_Detection/final_runs")   # ".../final_runs_smoke" for the dry run
FIG_DIR = RUNS_ROOT / "figures_compliance"
FIG_DIR.mkdir(parents=True, exist_ok=True)

CONF_THRESHOLD = 0.25                              # operating point of the alert (see the threshold plot below)
VIOLATION_CLASSES = ["no-helmet", "no-vest"]       # any detection of these = violation alert

MODEL_ORDER = ["YOLOv6n", "YOLOv7-tiny", "YOLOv7", "YOLOv8n", "YOLOv10n"]
SPLIT_ORDER = ["split_60_20_20", "split_70_15_15", "split_80_10_10"]
SPLIT_LABEL = {s: s.replace("split_", "").replace("_", "/") for s in SPLIT_ORDER}
COLORS = {"YOLOv6n": "#4C78A8", "YOLOv7-tiny": "#F58518", "YOLOv7": "#E45756",
          "YOLOv8n": "#54A24B", "YOLOv10n": "#B279A2"}

# ---------- 1. ground truth: one compliant / violation label per test image ----------
gt = pd.read_csv(RUNS_ROOT / "test_ground_truth.csv")
gt["violation"] = (gt["no_helmet"] + gt["no_vest"]) > 0
gt["compliant"] = (~gt["violation"]) & ((gt["helmet"] + gt["vest"]) > 0)
n_all = gt.groupby("split").size()
gt = gt[gt["violation"] | gt["compliant"]].copy()
gt["y_true"] = gt["violation"].astype(int)

print("TEST IMAGES USED PER SPLIT")
dist = gt.groupby("split")["y_true"].agg(images="size", violations="sum")
dist["compliant"] = dist["images"] - dist["violations"]
dist["violation_%"] = (100 * dist["violations"] / dist["images"]).round(1)
dist["ignored (no helmet/vest info)"] = n_all - dist["images"]
print(dist.reindex([s for s in SPLIT_ORDER if s in dist.index]).to_string())

# ---------- 2. score every finished run ----------
runs, missing = {}, []
for m in MODEL_ORDER:
    for s in SPLIT_ORDER:
        f = RUNS_ROOT / m / s / "test_predictions.csv"
        if f.exists():
            runs[(m, s)] = pd.read_csv(f)
        elif (RUNS_ROOT / m).exists():
            missing.append((m, s))
if not runs:
    raise RuntimeError("No test_predictions.csv found. Run the training notebook (updated version) first.")

records, curves = [], {}
for (m, s), pred in runs.items():
    d = gt[gt["split"] == s].copy()
    score = pred[pred["cls_name"].isin(VIOLATION_CLASSES)].groupby("image")["conf"].max()
    d["score"] = d["image"].map(score).fillna(0.0)            # highest violation confidence in the image
    y, sc = d["y_true"].values, d["score"].values
    yhat = (sc >= CONF_THRESHOLD).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0, 1]).ravel()
    div = lambda a, b: a / b if b else np.nan
    rec, spec = div(tp, tp + fn), div(tn, tn + fp)
    prec = div(tp, tp + fp)
    both = len(set(y)) == 2
    records.append({"model": m, "split": s, "images": len(y), "TN": tn, "FP": fp, "FN": fn, "TP": tp,
                    "accuracy": (tp + tn) / len(y), "balanced_acc": np.nanmean([rec, spec]),
                    "precision": prec, "recall": rec, "specificity": spec,
                    "f1": div(2 * prec * rec, prec + rec) if not np.isnan(prec) else np.nan,
                    "roc_auc": roc_auc_score(y, sc) if both else np.nan,
                    "pr_auc": average_precision_score(y, sc) if both else np.nan})
    curves[(m, s)] = (y, sc)
res = pd.DataFrame(records)
models = [m for m in MODEL_ORDER if m in set(res["model"])]
splits = [s for s in SPLIT_ORDER if s in set(res["split"])]

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"axes.grid.axis": "y", "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SHADES = sns.color_palette("Blues", len(splits) + 2)[2:]

# ---------- FIGURE 1: compliance accuracy, precision, recall, F1 ----------
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, (col, title) in zip(axes.ravel(), [("accuracy", "Accuracy (compliant vs violation)"), ("precision", "Precision (violation alerts that are real)"),
                                           ("recall", "Recall (real violations caught)"), ("f1", "F1-score (violation)")]):
    pv = res.pivot(index="model", columns="split", values=col).reindex(index=models, columns=splits)
    w = 0.8 / len(splits)
    for j, s in enumerate(splits):
        b = ax.bar(np.arange(len(models)) + j * w, pv[s].values, w, color=SHADES[j], label=SPLIT_LABEL[s])
        for r, v in zip(b, pv[s].values):
            if not np.isnan(v):
                ax.text(r.get_x() + r.get_width() / 2, v + 0.01, f"{v:.2f}", ha="center", fontsize=8)
    ax.set_xticks(np.arange(len(models)) + w * (len(splits) - 1) / 2); ax.set_xticklabels(models)
    ax.set_ylim(0, 1.08); ax.set_title(title, fontweight="bold"); ax.legend(title="train/val/test", fontsize=8)
fig.suptitle(f"Helmet + vest compliance on the test set (alert threshold = {CONF_THRESHOLD})", fontsize=15, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(FIG_DIR / "1_compliance_metrics.png", dpi=200, bbox_inches="tight"); plt.show()

# ---------- FIGURE 2: confusion matrices (rows = split, columns = model) ----------
fig, axes = plt.subplots(len(splits), len(models), figsize=(3.6 * len(models), 3.3 * len(splits)), squeeze=False)
for i, s in enumerate(splits):
    for j, m in enumerate(models):
        ax = axes[i, j]
        r = res[(res["model"] == m) & (res["split"] == s)]
        if r.empty:
            ax.axis("off"); continue
        r = r.iloc[0]
        cm = np.array([[r.TN, r.FP], [r.FN, r.TP]], dtype=float)
        pct = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
        annot = np.array([[f"{int(cm[a, b])}\n({pct[a, b]:.0%})" for b in range(2)] for a in range(2)])
        sns.heatmap(pct, annot=annot, fmt="", cmap="Blues", vmin=0, vmax=1, cbar=False, linewidths=1, ax=ax,
                    xticklabels=["Compliant", "Violation"], yticklabels=["Compliant", "Violation"])
        ax.set_title(f"{m} | {SPLIT_LABEL[s]}", fontsize=10, fontweight="bold")
        ax.set_xlabel("Predicted" if i == len(splits) - 1 else ""); ax.set_ylabel("Actual" if j == 0 else "")
fig.suptitle("Confusion matrices (percent = share of each actual class)", fontsize=14, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(FIG_DIR / "2_confusion_matrices.png", dpi=200, bbox_inches="tight"); plt.show()

# ---------- FIGURE 3: ROC and precision-recall curves (threshold independent) ----------
fig, axes = plt.subplots(2, len(splits), figsize=(5.2 * len(splits), 9), squeeze=False)
for j, s in enumerate(splits):
    for m in models:
        if (m, s) not in curves:
            continue
        y, sc = curves[(m, s)]
        if len(set(y)) < 2:
            continue
        a = res[(res["model"] == m) & (res["split"] == s)].iloc[0]
        fpr, tpr, _ = roc_curve(y, sc); axes[0, j].plot(fpr, tpr, lw=2, color=COLORS.get(m), label=f"{m} (AUC {a.roc_auc:.2f})")
        p, r_, _ = precision_recall_curve(y, sc); axes[1, j].plot(r_, p, lw=2, color=COLORS.get(m), label=f"{m} (AP {a.pr_auc:.2f})")
    axes[0, j].plot([0, 1], [0, 1], "--", color="grey", lw=1)
    axes[0, j].set(title=f"ROC | {SPLIT_LABEL[s]}", xlabel="False positive rate", ylabel="True positive rate")
    axes[1, j].set(title=f"Precision-Recall | {SPLIT_LABEL[s]}", xlabel="Recall", ylabel="Precision", ylim=(0, 1.02))
    for ax in axes[:, j]:
        ax.grid(True, alpha=0.3); ax.legend(fontsize=8, loc="lower right")
fig.suptitle("Violation detection quality across all alert thresholds", fontsize=14, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(FIG_DIR / "3_roc_pr_curves.png", dpi=200, bbox_inches="tight"); plt.show()

# ---------- FIGURE 4: how sensitive is F1 to the alert threshold? ----------
taus = np.linspace(0.05, 0.9, 35)
fig, axes = plt.subplots(1, len(splits), figsize=(5.2 * len(splits), 4.2), sharey=True, squeeze=False)
for j, s in enumerate(splits):
    ax = axes[0, j]
    for m in models:
        if (m, s) not in curves:
            continue
        y, sc = curves[(m, s)]
        f1s = []
        for t in taus:
            yh = sc >= t
            tp_, fp_, fn_ = ((yh & (y == 1)).sum(), (yh & (y == 0)).sum(), ((~yh) & (y == 1)).sum())
            f1s.append(2 * tp_ / (2 * tp_ + fp_ + fn_) if (2 * tp_ + fp_ + fn_) else 0)
        ax.plot(taus, f1s, lw=2, color=COLORS.get(m), label=m)
    ax.axvline(CONF_THRESHOLD, color="grey", ls="--", lw=1)
    ax.set(title=SPLIT_LABEL[s], xlabel="Alert confidence threshold", ylim=(0, 1.02)); ax.grid(True, alpha=0.3)
axes[0, 0].set_ylabel("F1 (violation)"); axes[0, 0].legend(fontsize=8)
fig.suptitle("Threshold sensitivity (dashed line = CONF_THRESHOLD used above)", fontsize=14, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.93])
fig.savefig(FIG_DIR / "4_threshold_sensitivity.png", dpi=200, bbox_inches="tight"); plt.show()

# ---------- Leaderboard ----------
board = res.copy()
board["split"] = board["split"].map(SPLIT_LABEL)
board = board.sort_values(["split", "f1"], ascending=[True, False]).reset_index(drop=True)
board.to_csv(FIG_DIR / "compliance_leaderboard.csv", index=False)
metric_cols = ["accuracy", "balanced_acc", "precision", "recall", "specificity", "f1", "roc_auc", "pr_auc"]
try:
    from IPython.display import display
    display(board.style.format({c: "{:.3f}" for c in metric_cols}).highlight_max(subset=metric_cols, color="#c6efce").hide(axis="index"))
except Exception:
    print(board.round(3).to_string(index=False))

best = res.loc[res["f1"].idxmax()]
print(f"\nBest run by violation F1: {best['model']} on {SPLIT_LABEL[best['split']]} -> F1 {best['f1']:.3f}, accuracy {best['accuracy']:.3f}")
print("Mean accuracy / F1 per model across splits:")
print(res.groupby("model")[["accuracy", "f1", "roc_auc"]].mean().reindex(models).round(3).to_string())
if missing:
    print("\nNo test_predictions.csv yet for:", missing)
print("\nFigures saved in:", FIG_DIR)
